# EPOCH 5th Datathon 본선 — Team3 빡빡이 최종 제출 코드

**최종 제출 = FINAL21**: `try_shot2` 블렌드(mate 0.70 + T01 0.20 + q_joint 0.10, 순위 블렌딩) 위에
**"지난주(08-25) 정답 1 + 08-25~08-31 롱폼 업로드 0편"** 채널을 최상위로 올리는 규칙 1개를 적용.

| 구성 | 설명 | 단독 LB |
|---|---|---|
| mate | 팀원 구조 시뮬레이션(sim.py) 0.8 + T01 0.2 | 0.7395 |
| T01 | ranklearn 모델 | 0.7032 |
| q_joint | 로지스틱 + 순위변환 + 우측 중도절단 소프트 라벨 (`run_final.py --lag1feat --lag1q --q_joint`) | 0.7115 |
| blend_90 | mate 0.9 + q_joint 0.1 | **0.7412** |

**규칙 근거 (테스트와 같은 구조 백테스트: 다음 창 26일, 직전 라벨 = 7일 전 18일 창, 07-25~08-05의 12개 기준일)**
- 지난주 정답 1인데 관측 구간(직전 7일)에 롱폼 0편 → 그 정답은 기준일 이후 영상으로 결정됨 → 그 영상들은 테스트 창 안에 포함
- 해당 그룹의 다음 정답 양성률 **77.5%** (31/40), 전체 평균 19%, 지난주 정답 1 전체 50%
- 시뮬레이션 점수에 같은 규칙 적용 시 AP +0.007 (12일 중 8일 개선)

**실행에 필요한 것: `./epoch_data/` 폴더(sample_submission, train_labels, video_5d_views)뿐.** 1단계 블렌드 결과는 노트북에 내장되어 있음.
(선택) 모델 출력 `mate.csv`, `blends/T01_ranklearn50_S01x50.csv`, `submit_qjoint.csv`
랜덤 요소 없음(결정적). `pandas`, `numpy`만 사용.

In [ ]:
import os, numpy as np, pandas as pd
DATA = './epoch_data'
MATE, T01, QJ = 'mate.csv', 'blends/T01_ranklearn50_S01x50.csv', 'submit_qjoint.csv'
D_TEST, D_PREV = '2026-09-01', '2026-08-25'
def clean(df):
    df.columns = [c.strip().lstrip('\ufeff') for c in df.columns]; return df
ss = clean(pd.read_csv(os.path.join(DATA, 'sample_submission.csv')))
order = ss.row_id.astype(str).tolist()
print('rows', len(order))

In [ ]:
# 단독 실행용: 1단계 블렌드 결과(try_shot2, 694행)를 노트북 안에 내장
# (mate.csv / T01 / submit_qjoint.csv 가 없어도 이 노트북만으로 최종 제출을 재현)
import io
TRY_SHOT2_CSV = '''row_id,prediction
UCl-VnCUwSxDr5zPu-YlDUYw_2026-09-01,0.11597364
UCnkytUgy0CtWd06up9CjNxg_2026-09-01,0.00179795
UCpI-AhCT02iQRGLC2ZcY0Jw_2026-09-01,0.00358407
UC9otskL_kd-0CDib_5Lj-mQ_2026-09-01,0.11835831
UCSzHok6X5qXEO7cjvVnE62g_2026-09-01,0.00949134
UCWU2V62eLE3oQ7AjIGS2STQ_2026-09-01,0.24379432
UCSfhr4gDPFsKuwdmTOWp2jw_2026-09-01,0.82712941
UCnhGcri8x6xhehanhAe9ipw_2026-09-01,0.00535579
UCoH2pCzAr6zWDG-LtUiLWRw_2026-09-01,0.01734613
UCUt2AEhC-OOC2EPCzj4DkCw_2026-09-01,0.28521633
UCrfB1-zWijAYkgfZW7Ehc8Q_2026-09-01,0.00272021
UCBTsPtRCh6ln3IRQFjhjh1A_2026-09-01,0.00171707
UCUbOogiD-4PKDqaJfSOTC0g_2026-09-01,0.00284821
UCnCH3Uhv_ynhTNKEH6ekRtg_2026-09-01,0.15188601
UCqIwPYLfMd3EoWYwvyTjPnA_2026-09-01,0.00856402
UCY_vDlvG-BTDRIOk_I9QQ6g_2026-09-01,0.71969304
UCZ-UuK86Bk5sGzQQ2cEglJQ_2026-09-01,0.00256825
UCv1tOT645xHOJrwdGlzjMaA_2026-09-01,0.01139184
UCuWDza8A_ntbjwBZgW0ZSNw_2026-09-01,0.74234895
UCrXfM4HsibSucC_NlIY7v0Q_2026-09-01,0.76377473
UCYGF_xk5RDpltyZDSx040uw_2026-09-01,0.01942432
UCZudBEEhIf6i8XIZZn3QDvA_2026-09-01,0.00573713
UCbFzvzDu17eDZ3RIeaLRswQ_2026-09-01,0.00175705
UCrY2B2XX41ReYh9DIWrFfCg_2026-09-01,0.011923
UCyiKngyUPUb1bka4yFrn65g_2026-09-01,0.80824035
UCybfHuHkJRzGDymek2Dsqvw_2026-09-01,0.00370972
UCEuP8zMn3ln3ZqJa85dp9JQ_2026-09-01,0.04693359
UCvPDPTxTLitVEyGu4Xt6m8g_2026-09-01,0.10147156
UCDHGqgpiHQAsE0UiltRqhfQ_2026-09-01,0.00628788
UCJoWNCiFuiN4eZnB80WUf4Q_2026-09-01,0.00529473
UCLHwMFguUxHVSk91vzicEJw_2026-09-01,0.01825282
UCRpdlPk671uOMiBtf5HtB3Q_2026-09-01,0.02406603
UC5grrciAyUF4xjd42XhV-1Q_2026-09-01,0.34869422
UC783XWULPW-nDGv_0EYlB6Q_2026-09-01,0.00456123
UCJpAwvQaZyCI5spAz7tipGA_2026-09-01,0.00223721
UCNmN3W2H0r5_hV2drzbEn7Q_2026-09-01,0.01016413
UCi5yzN6jpdz2akGTm-mJSKQ_2026-09-01,0.31797091
UCjyXK9pWHwWR_-bRibFaWDg_2026-09-01,0.008762
UCZPfs7xIjGmOGs9kvgGhUKQ_2026-09-01,0.44774197
UCepHesz_5Lwr7qRaqjB-p1A_2026-09-01,0.09143371
UCfIUuMA7U_jImP7oWfTe4Pw_2026-09-01,0.00275166
UCby0vMKcLR05ZdS90aZd5og_2026-09-01,0.78000681
UCQJuZxeDv2P05YHcC-tIEVQ_2026-09-01,0.05530283
UCp_jHNjcoiXC7SlElrBmzBA_2026-09-01,0.20382844
UC-2wa6jvprl7hfCpvw0ULzg_2026-09-01,0.05651973
UCvPGGL-OTEAQlwAUxbcJO8g_2026-09-01,0.00183981
UCEg24oF81p-THPtp3otdmTA_2026-09-01,0.00342312
UC-yb5eNBQyXBE9nZnht9u-g_2026-09-01,0.47782436
UCCMQyaKDjQJY79VimTdaYYQ_2026-09-01,0.00234253
UCO94iFhOhtdGb4Ze5DUAtbw_2026-09-01,0.15040705
UCQfd-b9GRBrUP0-wCqabLTA_2026-09-01,0.35131671
UCWH9vqm4yM9MHiSUPTfdJYg_2026-09-01,0.69829524
UCDBAVzfX3yZ1hah0FHnOoaA_2026-09-01,0.00190442
UCD2YO_A_PVMgMDN9jpRrpVA_2026-09-01,0.00204052
UCDYAog4sNhwQ5pXtAZiT0Og_2026-09-01,0.03680744
UCXS55IczImEf64aU16NzKlg_2026-09-01,0.03933961
UCab17MxxnFtAhGsJVTkSkCg_2026-09-01,0.32299154
UCQuY08EEKLFOp0D37Sfpd0A_2026-09-01,0.13899874
UCeiDuGZqWT4iiIXRTfhM3qQ_2026-09-01,0.0701467
UCdcjduaR6hyGHVNmpbJzSsw_2026-09-01,0.28170437
UCBvKuSAkXERBnGSQOH0Sarw_2026-09-01,0.01856526
UCdb5lyq0tWbY3jqH35SufmA_2026-09-01,0.00799629
UCaHIZAKows704_gk6bjdAog_2026-09-01,0.00216132
UC5RxNUZ9_c94ZEfJGnJZmTw_2026-09-01,0.79729167
UCb5NLtXAsTBrmaZVhyFa-Wg_2026-09-01,0.00323202
UC7nIK3KpkhbGee5K_sVBscQ_2026-09-01,0.01794553
UC7bcRlidTERRb_MKRHPVSnw_2026-09-01,0.00201718
UC4ncZM9bnBknxPkJbq4miHQ_2026-09-01,0.12576821
UC3Z9lUjG5BU60sce2-Axbjw_2026-09-01,0.21141344
UC3Oe-jfrIqEGygxYBYyN6jQ_2026-09-01,0.12450623
UCBtJuwDt6LlXp2YASC-ZMdA_2026-09-01,0.23539436
UC3hCAmwyYNgRuOOSFV7evVA_2026-09-01,0.03763358
UC3pTpoDKt17IrCDH60kSNPA_2026-09-01,0.59341428
UCkd3KAYGnTNPJZZrPOFeWhA_2026-09-01,0.00927703
UCsmsw8102lekPyVYwKMHxmA_2026-09-01,0.63970025
UCtLWxhEMKOio-GzdVJ1CVhg_2026-09-01,0.77199254
UCkrNv6xF8j7QZmWb3UeAH8A_2026-09-01,0.68849305
UCkoszQp0YiJa9lGm_wK0xGQ_2026-09-01,0.1928339
UCraf4ugZTyvFY6AJ0VjIxFA_2026-09-01,0.22719686
UCsuanL5kXO29Uzz3MWlBsMQ_2026-09-01,0.70071825
UCtGMgPWJNU2Aj-SNK0zg0Sg_2026-09-01,0.81002061
UCrExV5uWnC3-WUnUN2Ck8HA_2026-09-01,0.693416
UCqyzJW0VRYRP_sFELZYdxGg_2026-09-01,0.1689505
UCK1aVlIbEBrdhv-gbSgGIkQ_2026-09-01,0.71736168
UCKrprPfevcAZpPuRnCO6xOA_2026-09-01,0.03220552
UCMAfHuocszM6UTORPwUmeNw_2026-09-01,0.64499671
UCM81CAJ2_Y42-bvK1OQnAAA_2026-09-01,0.00305155
UCJJZkVUNIxq8XNDzHiAD-eQ_2026-09-01,0.00763871
UCcstDLbuhxEbFKkPZ6wST2A_2026-09-01,0.01920597
UCULRVjf5z_WoR106msn8HoA_2026-09-01,0.01126275
UCcJliZnsRbNw4I4uWSGq_wA_2026-09-01,0.44204835
UCJmnOEp1-OITEU0mk4icM5A_2026-09-01,0.00192646
UCZ6mNZTiTUrUSbSYPlclI4A_2026-09-01,0.19644778
UCp6n8d8Y8r3MwKNw_MMaouQ_2026-09-01,0.27706287
UCeOXoe4zJCBTBwHN1M_djUw_2026-09-01,0.31547629
UCgO8sNnksVuKcuTVTQKSK3A_2026-09-01,0.0056071
UCgLFbrC7-g0ihBKB0obVPTg_2026-09-01,0.81178819
UCJDXc9JWCZo-nLNKr_-L2Ng_2026-09-01,0.07400055
UChHxLcNi7e0zu-e33HM6v6A_2026-09-01,0.06939853
UCwfbSsZ7_sccxcibe60mWlQ_2026-09-01,0.11132938
UCifCesg-EUkjKyQedaB3hRg_2026-09-01,0.28521633
UCcgkgKhXT6QwdGj9Nr8PBcg_2026-09-01,0.00713188
UCdQHEqTxcFzjFCrq0o4V7dg_2026-09-01,0.15946048
UCcHVk5SFxfYhCbbX8pWWuVg_2026-09-01,0.08057336
UCwBkHOTyu57NQi0syf5MLig_2026-09-01,0.00477544
UCfcfymsN3OMo2cfmbpGTanw_2026-09-01,0.09434745
UCfjMjhfjRRy8B0mfqbO_43w_2026-09-01,0.0650626
UCjSNmS8qcD1jEdQhU5h-zKQ_2026-09-01,0.71501862
UCww3MeqGQnIatVvEr3pSjhA_2026-09-01,0.02816214
UCwlUB3goOsh4tpqZKIzUAbA_2026-09-01,0.02632971
UCjUnlBpn0Z1L450g79fniRg_2026-09-01,0.8064474
UCz4b1O9sT73P8DWwx6OvQwg_2026-09-01,0.01165443
UCVey3Q_CodrHAWBonTGbHFg_2026-09-01,0.22922718
UCVnp5Jumir-XIyeejktUhlg_2026-09-01,0.00746591
UCPuD5ZRQWa_JCaY5NokNhtg_2026-09-01,0.42506441
UCeRn_02zEifLqXxZkTpiXzA_2026-09-01,0.51094926
UCikKJvpU0Ylua4hRW6faQRg_2026-09-01,0.24167543
UCZNTsLA6t6bRoj-5QRmqt_w_2026-09-01,0.00346267
UCy8OX8RrstkjRrnM5hND_ug_2026-09-01,0.42788393
UCyZXLzORHoh9QMwA7IwS0pA_2026-09-01,0.83201839
UCvrxJgD0T_TFvgsO2nA-fDQ_2026-09-01,0.0024528
UCxpXguPL_B5pz7Gq8ozfwlA_2026-09-01,0.68352713
UCz5UFGucIyJGH0nf_8d0ghA_2026-09-01,0.20196414
UCYG8tuvbQzWsDeoADyJFU8Q_2026-09-01,0.32551733
UCuZ862fZel1gq9RAf_OkvqA_2026-09-01,0.00755182
UCzVXdulvpgEoI8yuFMmhxJw_2026-09-01,0.1322433
UCvuc6-wCcCeDrQ9dA52pBBQ_2026-09-01,0.62631179
UCuFguUhr9Wr0tFY85nDGk9w_2026-09-01,0.13092607
UCzps05UZ7XpX3SFoFyYIDow_2026-09-01,0.01620573
UCYJ0Ucu9jPX5kn6SeDcNaIQ_2026-09-01,0.00383975
UCQSEAbOs6vsJfy7WN7iYaGQ_2026-09-01,0.45916894
UC00raQBBBGagQthJ01ZYIIQ_2026-09-01,0.6181833
UCJzfS-WmieHog0H2OQoXkeQ_2026-09-01,0.15487974
UCnju7ULBic4VgkCfDvVXCnA_2026-09-01,0.01566351
UCNNR1-x2NhZARsNGnQlh2sQ_2026-09-01,0.00600623
UC-S8JFhLkQ5enhB70c2CYIw_2026-09-01,0.00169742
UC0PMQXAwF6O6aeTpv962miA_2026-09-01,0.04491363
UC0aKwoKNeqBaUwiEXmkQaGQ_2026-09-01,0.00772659
UC2WBkFH1nHcO6aFLWo6sZQg_2026-09-01,0.01100888
UC0tmAwhgk_1ATJtFyVHrFxg_2026-09-01,0.01531199
UC24GgPBq5W3v34z0p0uk1Ow_2026-09-01,0.75106919
UCKUUL3fMVvYVZdGkKhhjP0A_2026-09-01,0.01004881
UCL2ZLyAV0H9Pvi46W7TGdHQ_2026-09-01,0.02199257
UCYJDUekoQz0-bo8al1diLWQ_2026-09-01,0.00173694
UCaY7wcqhFb_c2eQ1eRB8rOQ_2026-09-01,0.00587015
UCnMshUhBxDsKFPnxBGmGEpg_2026-09-01,0.76168861
UCjsqgRBvkn8XadWraSQxTKQ_2026-09-01,0.79542226
UCgeazIBHz3JS7ft7_nyj-0A_2026-09-01,0.5396858
UCfq-5-0b5Lzj7cKCrp6-flA_2026-09-01,0.65025787
UCjiPEaapiHbJMoAdi_L8fNA_2026-09-01,0.00896452
UCjBPx2oip2bKnnfqKyas0ew_2026-09-01,0.67092947
UCiE9ZR51IeGPRbD4_Px6_Fw_2026-09-01,0.00301668
UChKYbZ9fSz8DMM6wU-t1lMQ_2026-09-01,0.12201484
UC86GXx7Ap-Cahlb0r0xT-ZQ_2026-09-01,0.09533704
UCd6CX2LiQE2dEAPXwk2N0jg_2026-09-01,0.01774351
UCQzgMwT5Q6x-Mtq65HRS_0Q_2026-09-01,0.23747542
UC9vz3TBChrJmYbzUo-rqfSg_2026-09-01,0.01398195
UCSh__r5HS4Y1xBkccWnV0TA_2026-09-01,0.00226308
UCS-X1uBDTNwAYyydiXjcX9A_2026-09-01,0.62900576
UCSkCNddZwo__x-iUr-KDFWw_2026-09-01,0.00397432
UCSl8TBqqflby-ymdEQVfRNQ_2026-09-01,0.0055432
UCOgXOOKbZRtfQvlggBpN6zw_2026-09-01,0.80282334
UCcum1rCJ5GJeQ_xv0xrohqg_2026-09-01,0.00960033
UC_RRZdJk55ShXE-9d3deRUA_2026-09-01,0.01254984
UCPN9TrChCye1s97iEUOsIeA_2026-09-01,0.14604119
UCBo7GIv99qTzHaa6ip09YLA_2026-09-01,0.57662497
UCTmniSiCVyG_UKVCEa9PNKA_2026-09-01,0.37530838
UCamHctB9_ZaFO9vZElnl_Yw_2026-09-01,0.58504431
UCa4clnMsHg6uJPpxKbxCEmA_2026-09-01,0.00650769
UCarBV0zWlC2k-HteyeKqfag_2026-09-01,0.00705072
UC3RzmV2LX3AmWhZVx0A7u0A_2026-09-01,0.03640094
UCeBa5aBvHvly34S0Uk9Wqig_2026-09-01,0.76995713
UCaj7p2XHGqfBYb2OI-7Q2DA_2026-09-01,0.01291203
UCDLDbittWhk2KXvaCT0YpZQ_2026-09-01,0.03599876
UCD8r5Jgns2V64-NcUmXclpg_2026-09-01,0.00580326
UCtUHgx7SA_eZFssVODDh82A_2026-09-01,0.00511568
UCkphwoGMBs2_0raLOqu9aQw_2026-09-01,0.81528539
UCm6Y-gVqhlhX4FzrJCtsSzg_2026-09-01,0.01366764
UCmcBe97VA2LqVEQUeeZl6Ew_2026-09-01,0.00499967
UCEcRvF_6IRSux8FO-ogFZug_2026-09-01,0.00614545
UCGW76VChAJKee9kYzvyoycQ_2026-09-01,0.43636993
UC5WN-znPsJK0BbA8aHxZHWQ_2026-09-01,0.05902941
UCaova9Q4UZLPqFnSvq8VxzA_2026-09-01,0.0167664
UCaN3tSrrzIrXSDonEe7LK9A_2026-09-01,0.05411062
UCHCeN9BUgogMGrHBC6zKYvA_2026-09-01,0.08405595
UC6JTm9ObvQFuTEOxOsbmYwA_2026-09-01,0.01877647
UC7hwQp3F6P_OwyRM14p88iQ_2026-09-01,0.00494265
UC7wtTnMorJi_d93lb_P2Xow_2026-09-01,0.01276594
UCqPN9LAkx20aBWOafwvQB-Q_2026-09-01,0.00221162
UCIsrVj53L57rcLAPOre4Xhg_2026-09-01,0.00461387
UCHbz27HAkCIQNw_G2Pa7XVg_2026-09-01,0.03011815
UCqVkQsNin4MdGFrWvzFVOKw_2026-09-01,0.02912422
UCta3D5-EYqO-pduQLxQ8mNQ_2026-09-01,0.07559596
UCqjUPx-UAFHVvthd71bY7_g_2026-09-01,0.21334166
UCAenntxQB3F7AH637vySglg_2026-09-01,0.36991888
UCATlfGN7EDjXS_2ofklb3bw_2026-09-01,0.00334536
UCvagJYsZ_uvB3jJoNwuimZQ_2026-09-01,0.53108378
UCrxzkAOXEbflxcTbRYnCoTw_2026-09-01,0.04250506
UCk3w4CQ_SlLH4V0-V6WjFZg_2026-09-01,0.0679243
UClHTjLVog7CkRF8tYJuNXXQ_2026-09-01,0.50374633
UC_O4_e0nbsMoU1rg7JU_nwQ_2026-09-01,0.00326937
UC3BUvshHmW_-ReQX1B_EeOA_2026-09-01,0.08767471
UCeB_ZeSo0xDL62wWlkTEtjA_2026-09-01,0.02102236
UCOaUTqrtz1yBBiwziVJEAmQ_2026-09-01,0.1079537
UC-aXNBQ6nwMNeW2V8XB7sBg_2026-09-01,0.70553063
UCXCa03lRK6tlntZoJ7TUWRw_2026-09-01,0.25457641
UCXQoI3upwKr47JAiVaYkvIw_2026-09-01,0.18575846
UCNymqCXr3sxL8dnmO_jScuA_2026-09-01,0.00993478
UC14G5tdSndPhP47RTVgMXcg_2026-09-01,0.30560463
UCJj2vuQIpFnk5pbTAtCi3yw_2026-09-01,0.03184816
UCMZFHqDjg1IE5kzm9fZBOFw_2026-09-01,0.01178796
UCc1VJX_5Z2RdGi58XU4IfbA_2026-09-01,0.81701505
UCT807kAu6fmwES8B5698e2w_2026-09-01,0.04066756
UCS7gEaMmBPAaI8dHNnrSofg_2026-09-01,0.04442173
UCcp8n8YZXg5qko87BgVwg1A_2026-09-01,0.0018611
UCzDyC6JMEXEufozffWzI9Mg_2026-09-01,0.49942363
UCZPBWThSlgWT0DVaxxUEX4A_2026-09-01,0.38345053
UCZuT6NXrfZzYbqCG_RWgSgA_2026-09-01,0.33059936
UCuWcjpKbIDAbZfHoru1toFg_2026-09-01,0.55965683
UCztGUZP5cCuIZDViaJanLKQ_2026-09-01,0.3105189
UCzSODDx0zQF3nxXOZISCm_w_2026-09-01,0.48214017
UCWjfI3wnNOgQCTu2SijfgMg_2026-09-01,0.24806973
UC1Il94cQwl93Zun8ax1Qjlw_2026-09-01,0.70791984
UC2OFe8XUEqDT3eJGO-0EDNQ_2026-09-01,0.48501889
UCv0cBz_Vdmd84xYl7gKWwQA_2026-09-01,0.33829683
UC7gozbdUqpQOYN1og_d-J9g_2026-09-01,0.08584806
UCw55Gv3z7xgtG34fOP1-Aiw_2026-09-01,0.05839237
UCIXQkaLrM7_t87SEM_pmHpA_2026-09-01,0.36189541
UCA2ztZuXhSktpdxb419WvCQ_2026-09-01,0.64763177
UCAaLhok91QDrweyuDRsmjqQ_2026-09-01,0.0059378
UCA9f2-9fUklMMOaz8vKW7Jw_2026-09-01,0.60173035
UCP8Q-Ku5_RkANynqDFzDWnQ_2026-09-01,0.51382932
UCTVkjiEiaiSbEq0Qwd5XTPA_2026-09-01,0.82877146
UCAEQiQ4Mg6fFkPshBMhPaxw_2026-09-01,0.8009922
UCg80LjrsnWo0T3T3uYCPhrQ_2026-09-01,0.19104602
UCGi94VU356EAvQLgODywyrA_2026-09-01,0.11956649
UC9FRxVaiS2aY0ANaWLP8SMg_2026-09-01,0.34608086
UCHK-FRuAjk8g798dD7L7xqA_2026-09-01,0.07166546
UCH29Gz8a6veRLqUxzjBXHrQ_2026-09-01,0.00818126
UC9zmo9x1o7xAfgM1Fw3Dg4g_2026-09-01,0.72661617
UCcJ_oPUSkS_h96_PuizxMGA_2026-09-01,0.47782436
UCYTQCmuFcacJJ_Tad20qqEw_2026-09-01,0.01039872
UCXqP0RsJo6-y-MrKz6k8H-g_2026-09-01,0.70313008
UCL3ooD3l7KEqb85EBZSEkZQ_2026-09-01,0.01305978
UCtjBNJ4UbuWGLe-415urMOw_2026-09-01,0.00665845
UCzT9VqGDsCxleSmXzJ80hBQ_2026-09-01,0.12962
UClpc-7ejy_cntRdmrFLbOfg_2026-09-01,0.09336708
UCn3lLusvOTVdtxZi0EHlSTw_2026-09-01,0.81873212
UCoovEJW0dyTWwHiblWDWvyQ_2026-09-01,0.13491145
UC2IYsRiATGe71sLIOlSXGuw_2026-09-01,0.63436957
UC2f8VfmEF2_0-rFN4RNHaNg_2026-09-01,0.03847752
UCJENagCOfjMxdQUDxO4kq2Q_2026-09-01,0.74454759
UCklDsqQMGPYIMpIhQ5Jz6jA_2026-09-01,0.02009407
UCI4wEWWBEElIbp9jHcFllTw_2026-09-01,0.43353687
UCE3q8VKqIT_vOL4b9Nl8chA_2026-09-01,0.09239585
UC1p5kp09LAS3kpQB3O8YQJw_2026-09-01,0.79354007
UCIzbqIyEUxtFLYh_sCDoXpw_2026-09-01,0.02126091
UCh7XUfZEEV3pct5zDjx6OZA_2026-09-01,0.59063006
UCiVO_mThS9B3tNc_kzpn-3g_2026-09-01,0.02880004
UC0E_zsPAziGgkxaAtwvmFAA_2026-09-01,0.00435659
UCKHx_yf3dI2ajWAjXRJ_9Rw_2026-09-01,0.40825555
UCKiCT33Gmj3mlJ5xh-9ZUqw_2026-09-01,0.07321451
UC2t5bjwHdUX4vM2g8TRDq5g_2026-09-01,0.04958011
UCKmOaDLQpgbsmGsHpV9688g_2026-09-01,0.29831567
UCPPuXI31ybWYB560T2uAeMg_2026-09-01,0.14893995
UCMgTCM5mGx3y4Hg0aMOXrbw_2026-09-01,0.20949802
UC_EEqHc75kYwzrN7L8T2Bfg_2026-09-01,0.16573813
UCsii9gve3wx7kHDfVlM2Stw_2026-09-01,0.0230065
UC7wosSPUA3os3b3-RMZZe6g_2026-09-01,0.38617942
UCC1veMJY2wElX0lpYOoNB1Q_2026-09-01,0.00294813
UC46X_E7jxvJ1O_3gnaeCxeg_2026-09-01,0.00253889
UCkmUJzaZfP7Wsb5-89pAK4Q_2026-09-01,0.29590834
UCkpKYfb8wM67QnKxBj9bZXA_2026-09-01,0.03149464
UCktXAh1Sd7wf8eQHc029-8g_2026-09-01,0.11479704
UCql_fqTxryFKW7jukEvNoCQ_2026-09-01,0.28875448
UCIQEX0V3QxP-tY_jpRT-waQ_2026-09-01,0.20759538
UCFvdXoI9LTADYG07s4pMnhg_2026-09-01,0.03114492
UCHoo90P0hvh2rrbGvX53G0A_2026-09-01,0.41663575
UCms8Ge7H1ZGgw6nV94fUhUQ_2026-09-01,0.08953643
UCtKNP4UGUA1ybxJJcTC_GFA_2026-09-01,0.06647915
UCFiHjGyrJux8Rt4e6mUPU-w_2026-09-01,0.02274882
UC9M8GGVwPqc9ubNruArCNPg_2026-09-01,0.12832504
UCnDcpS7XLQ9KFPqULb9TH8g_2026-09-01,0.00548001
UCxDq2kGOt3h2_ilI0Wih5dw_2026-09-01,0.78588348
UCBkvEgeJ3_UMwbW7VCxzs8g_2026-09-01,0.35658857
UC0wo7SwsC5dE0nSgBs2M-3A_2026-09-01,0.00430687
UCzHnjIeMxo1gViogzN9xMLw_2026-09-01,0.27475989
UCaN1Qp37sHiCdQuSaXa3fVw_2026-09-01,0.68102829
UCtIbfg_6gjbdFqCqAiUcqyQ_2026-09-01,0.72201261
UCEJF0q0SqEJ64uA9LCXuSrQ_2026-09-01,0.80464173
UCaZv-jDn5Mq5KJ1buXDRd4w_2026-09-01,0.10906881
UCDmmbxGg8g-EBkC_ku6vybg_2026-09-01,0.55396762
UCpAZLM9XMzIbqFucUBD8Jtg_2026-09-01,0.64235283
UCe-7GwWL1XNlCJyfqBhgrHA_2026-09-01,0.02078643
UCnCeDIkExSpULGqmpj5oafg_2026-09-01,0.75535461
UC2YA1b6qKO2YteKYFuMiYPg_2026-09-01,0.37801486
UCW1FuNP_2X03y7H0HIujOTA_2026-09-01,0.05590817
UCGX5t-qAZjL4xl5xZ36Md5g_2026-09-01,0.24592577
UCvsw3aXHnikw3MVGSwOICxA_2026-09-01,0.26119448
UCHXhFmex5EkUy-fP2IMQ_Ug_2026-09-01,0.20570552
UCmvMUe1GqpZFkj1e-V532LA_2026-09-01,0.45059402
UClFSQxcVutJFEI9nCAOOkgQ_2026-09-01,0.51958651
UCXb8sP7slkJyunTEAsP-CiQ_2026-09-01,0.50374633
UCR0aLp4aIvxS07g6rGWZr_g_2026-09-01,0.07090232
UC6tlo9WajrgiX3Eej_jEr2A_2026-09-01,0.01584224
UC-StHq1pWM-t4XS9ycZpn9w_2026-09-01,0.04850494
UCbYO4SBywZ2oWv1u309oWmQ_2026-09-01,0.59619251
UC8MgGtsJazbztr5eGWiqUXw_2026-09-01,0.38891549
UCE-6-bWkikwoP4_ooZNep3g_2026-09-01,0.02174599
UC8CX0LD98EDXl4UYX1MDCXg_2026-09-01,0.07479434
UCKY2DOkLOyPLlvGHI3XQw0Q_2026-09-01,0.45630769
UCLXI7tT4pHH222QgpvYx1jA_2026-09-01,0.0246135
UCo77zdYpvhgUsdP0Z0vmz2g_2026-09-01,0.81354311
UCdZrsVZ3QolU1TWFaDgN2ww_2026-09-01,0.58784001
UCEjXNNB9IKEa4HfLQVt8VEg_2026-09-01,0.07804835
UC4VWaR3R17Ruc7j1Cy1-PKQ_2026-09-01,0.82380799
UC3ozN5tHi449NhCDE6Xtmwg_2026-09-01,0.0251731
UC3gl4CHfNtAqNjju1goB0_w_2026-09-01,0.79914832
UC3_QWAvQRhPW4CDoQU8acKw_2026-09-01,0.01051801
UCAJ-meoCh1TrPZ7La3UpPrw_2026-09-01,0.02574508
UCpcTrCXblq78GZrTUTLWeBw_2026-09-01,0.00208801
UCqI5lyTpC79pOy2D-VXAMdA_2026-09-01,0.00338402
UCPI40-XQA9-GBGj2k8PPqfg_2026-09-01,0.00281566
UC8FErYSi74YwGUAoTpjvgzQ_2026-09-01,0.55111761
UCpX86JOUhnrzG7MK8e8Gjbw_2026-09-01,0.00319509
UCAtGPbK4AWmeoaREsNLLZhg_2026-09-01,0.00375256
UC8rbhIem-3aAsNXhTDI-nSA_2026-09-01,0.00450919
UCBY9rOZPo0fLE45mOV0s0cw_2026-09-01,0.78781681
UCS3_H_feQRm4QbXipKTvSrw_2026-09-01,0.04345331
UCVZI7eLie2UlZph6aiBRNrw_2026-09-01,0.00402021
UCnV5tUnOVGyT-Ncj0s0tPTw_2026-09-01,0.00827533
UC8CIM3d3zDYMk-3T5aAz0yw_2026-09-01,0.00177738
UC8CZL4cvFiKwKKDNCuM2lTA_2026-09-01,0.66581946
UCo33r3SAf5z9MS-lrXV2aOQ_2026-09-01,0.00236962
UCoSniqkitCpxlHrQd5L8A1w_2026-09-01,0.25897615
UC9jOsXFn3RwXEW3qUyySjAQ_2026-09-01,0.33315537
UCBA9XaL5wCdHnC5EmEzwrqw_2026-09-01,0.00837047
UCpCBP5cVIBvezPI1LKFMYjA_2026-09-01,0.08229789
UCt8CJiye87FMbEwt0npPMuQ_2026-09-01,0.01479922
UCAl4HWznMn7KhKBRFO5eiFA_2026-09-01,0.02379677
UCYooECrv-fIXlPaDwEIaveA_2026-09-01,0.00425772
UCvAetSOU_4lyjM5hVMNzN0Q_2026-09-01,0.22517927
UCtt0Hg9MhraTkzWwNQpplHg_2026-09-01,0.67346949
UCrR_fhiJjoW8OJFzUcmlMMA_2026-09-01,0.00781547
UCuwnt_REmD4YsCtFJRoqjFw_2026-09-01,0.02945195
UCrKSaegnzewdj9-YLsUhoZw_2026-09-01,0.02150211
UCabScupufwcOzAdcVXZrmlg_2026-09-01,0.02353045
UCYsRymb9vAsOOLBy-Wo1kRg_2026-09-01,0.23127021
UCZRH94I89Ps7giJYeK6DlWA_2026-09-01,0.00315859
UC3DYSay178fMMPaf3zHwECA_2026-09-01,0.00697047
UCDbOXmGL7EkmOcnUb5Kki4A_2026-09-01,0.0133603
UC08-hQbwh7eyr8PlaYG2-sQ_2026-09-01,0.06576733
UC-8S20Ll1gzPiu_YY8BgyaQ_2026-09-01,0.00194875
UC1dG3vI9FfHnH3YgyeKUz_A_2026-09-01,0.00206413
UC2NFRq9s2neD_Ml0tPhNC2Q_2026-09-01,0.00211217
UCI6Ro_MOr36re4Bhb7nCY7g_2026-09-01,0.73791476
UC6JB_a6rD1lZ_aZgQvORfrQ_2026-09-01,0.78393737
UCKVt3arMXfa6XZezR1UnjqQ_2026-09-01,0.0963359
UCTK7GGeH0c9INAEnV2EfD0Q_2026-09-01,0.23956912
UC3hqiw3lZAXaMwhQvMGZkdw_2026-09-01,0.17221228
UCvUKUkIM67eWFbQemjj2-eA_2026-09-01,0.02784836
UCzRPPKLkNxjD2of-0Xxfnjw_2026-09-01,0.03405122
UCw-JzmPsjRcbzJLncr4pIIA_2026-09-01,0.00350267
UCXAXCxTCCYYKsLJNzEvjqfw_2026-09-01,0.00866245
UC699Y51pMl0wWLPA4atcZjQ_2026-09-01,0.03045671
UCyAA-Vj4ORjXl5Ui2m6_Llw_2026-09-01,0.00938358
UCzDmGDR5RdahQkLCBoWlTEg_2026-09-01,0.00738098
UCJYdIDfGkqXTsQq5aW4md0Q_2026-09-01,0.52533851
UCNu00qhYj4nPeMNdxuwV6sg_2026-09-01,0.05294268
UCPJGXrPIVKBR8PJY7w33WZw_2026-09-01,0.00673513
UCmiTZyj9GwMjyznZxrgCH4Q_2026-09-01,0.01548677
UCaMV-CjrWYh2Aefg_8iLwJQ_2026-09-01,0.00259795
UCYjNEoVziojh1ZCum1q0PNQ_2026-09-01,0.0973441
UCXVVG5ZM3SJJCh94xatR8Dg_2026-09-01,0.27937772
UCd-Ziz8pAMZBWl7juEghsag_2026-09-01,0.00689113
UCN9VGDvz2M3vdR0czEUp6lQ_2026-09-01,0.4054737
UCN8L7Z-dyzu9FzT39RM0fvw_2026-09-01,0.00846669
UC_yxsHiFh1_j174x4AxICSg_2026-09-01,0.06299036
UCg7rkxrTnIhiHEpXY1ec9NA_2026-09-01,0.00181876
UCgWwo8FJL3MCxfd66ip7nFg_2026-09-01,0.04203838
UCyoSQyaCrZHro6F_nBiXIYg_2026-09-01,0.00199411
UCvjzWMh2bihwlIN-WRtddkg_2026-09-01,0.82043661
UCeG0ONJtjwE20ThR5iSYdsA_2026-09-01,0.05179856
UCd9oeoY-jLSkhd-J5y_1aYg_2026-09-01,0.77401523
UCePy6AqRuTfKDC4aTrqcAqQ_2026-09-01,0.71029763
UCbeQg95cAxVRsxxh6FyilxA_2026-09-01,0.00643358
UCO8ToDSM2Gvt3EZdOdosxTw_2026-09-01,0.25239509
UCMswODzU1ZjjrLyUk7NfwPA_2026-09-01,0.07640546
UCMpvRMwoq_WmradKdysk6Cg_2026-09-01,0.00466712
UCXYNuWjnQRSAXCGl9gv8Ncw_2026-09-01,0.00388409
UCSgvQDek_7CQj08fu7kp1Iw_2026-09-01,0.007297
UC7loyhgCm1NRTtyF36hO-jA_2026-09-01,0.16415039
UC8x8GQmlxAIhHxzU61z0uLQ_2026-09-01,0.0415766
UCBW_PhuXoHk40xF6nO0Q64w_2026-09-01,0.6528749
UC9JlObZnbOirT4_iyPGrdcw_2026-09-01,0.00971055
UCB4YEVYp6LynBpz7bXyZsTg_2026-09-01,0.62360995
UCF-rFAdN7dzd-SI2T3rkCaQ_2026-09-01,0.04393497
UC027x98NgUBw9K4aPT8k0mg_2026-09-01,0.06367418
UCuCmLrNT5MbpqfrSSKHqTug_2026-09-01,0.10042537
UCCd3jyJmOFzkJEMj5Bp89rw_2026-09-01,0.03890627
UCGa52d8Qlv3THghuCAt_nXA_2026-09-01,0.18058458
UC-FiQR2gg8oiNw98ZdsSX6Q_2026-09-01,0.0111351
UCooLkG0FfrkPBQsSuC95L6w_2026-09-01,0.21528265
UCpn9dP1b5BhBsU9SduEz1hA_2026-09-01,0.01382391
UCCvEN8bTeF8Z2jcN513T1Gg_2026-09-01,0.08494772
UCBvkOBLsSj9kusGYlroQkBg_2026-09-01,0.00354314
UCQfISxq5legTqOt1h1us6lg_2026-09-01,0.69586113
UCdm06VHBWF5Mg5kYxffzikw_2026-09-01,0.66837943
UCQhIdo5nRQY9DB3uo7vawtw_2026-09-01,0.72890004
UCawK5EIfr66Z7fjAlZgfB2g_2026-09-01,0.06032308
UC4P7F8qenDo1O_5NvgBb_cg_2026-09-01,0.00218633
UC5H8Wt-1q_tnn-m4tDTqbUw_2026-09-01,0.07243617
UCQDGJQe8X9tbk_i8Na3VVXg_2026-09-01,0.77602517
UC8Xz3BKjDvQk7lKhH7Hi9nA_2026-09-01,0.04591303
UCF5RrlbsxJjAVLWgOCoNHMg_2026-09-01,0.17386179
UCC1LvVTX2ySKYjeIXkAtvsQ_2026-09-01,0.00298221
UCCW6uYIUP4YKKe50k-ktVAw_2026-09-01,0.00445774
UC5T_FfQwgy56XP0aEVHYULQ_2026-09-01,0.033301
UC60HSePkZwGFzRj6VSDa6Ew_2026-09-01,0.63169174
UC72PuhDwKtZ5MikpGNhPAtA_2026-09-01,0.57098694
UC5bjQxLK_3XVzIOTQ04sYeg_2026-09-01,0.77802237
UC6oq8RPEH1BmtyldQgXVbJw_2026-09-01,0.63703912
UC5ScPjbt-a97AJO7SBj6nAg_2026-09-01,0.00330715
UC43hOVGChFuQRV47vxc_c-g_2026-09-01,0.74890775
UC5Yx9RlmznXDUEIrfoSqheA_2026-09-01,0.56533042
UC6YqXiwCBfPvPTs3wXgWElQ_2026-09-01,0.02489176
UC543-TDm6IvHHZXwdSFioMw_2026-09-01,0.00278347
UCk5mlvpbj8CZxD229CpiouA_2026-09-01,0.72432034
UCk5C4AR3uvmKwZYkIlSobbg_2026-09-01,0.01219769
UCkcaRxpTcisadUeCbHOYTFw_2026-09-01,0.57943662
UCs0LzOveQMLImmmTrMmP2sg_2026-09-01,0.00213661
UCkavlo6q2XfO_MA9Q24wRvw_2026-09-01,0.71266391
UCl402YYy7RcH7PBI3npmGPQ_2026-09-01,0.00188263
UCs-JvUOu6kEEO0BcTFhUErw_2026-09-01,0.22317442
UCt15X5eHLwyP8PpNtQTkuDQ_2026-09-01,0.00721397
UCsRiYH-VrZ2mYsohk7GhLdQ_2026-09-01,0.0643649
UCt6Eo4xMTHhlOWg4_PAz_MA_2026-09-01,0.78973735
UCtRiSpvkhtRXaSjVjEB9LKg_2026-09-01,0.54254815
UCrD6viwK0DHbQuZSOZBbofg_2026-09-01,0.00790537
UCksGX0hODm1Ev6mysrToMqg_2026-09-01,0.14748468
UCV6mNrW8CrmWtcxWfQXy11g_2026-09-01,0.02753797
UCOTJtP0Z9DjpPU43B5m1V6A_2026-09-01,0.00517468
UCKttwyzsHvW7bCo7YfTacVQ_2026-09-01,0.41104328
UCK8iWEZupSV2pmDMxtwdAmA_2026-09-01,0.50806846
UCR1ctqL11HoxQkbSkoUAYeA_2026-09-01,0.56249567
UCKYqtXYbSqaEwBmX43Q13Tw_2026-09-01,0.61545878
UCMo4MnhHhSDkiUIXS6dytTQ_2026-09-01,0.05967296
UCw1DsweY9b2AKGjV4kGJP1A_2026-09-01,0.00917168
UCUR5w0E-jQTOO8Q0FZ4ilYQ_2026-09-01,0.001678
UCW_Ptq0v8ck67w9oKiwXz2g_2026-09-01,0.34347677
UCdF0eCyf5np3lPO4y1fJVdw_2026-09-01,0.18750833
UCJGww2K__Q3y-MtDi0XWH2w_2026-09-01,0.00411359
UCJZsABKjWUGbT0qf3MsXH8w_2026-09-01,0.612727
UCMF3M48BakI-ptofqoUqdAA_2026-09-01,0.13357174
UCZ49HaU_D0Tlf1aDA2jU_Vg_2026-09-01,0.00228926
UCoQU5wtIxq0QGh80NHky82g_2026-09-01,0.01657742
UCpTNtH8yc1Kdl_YFfsQsTYA_2026-09-01,0.04797569
UCdXpKf3aIenvWkMbF3EvcQg_2026-09-01,0.13626248
UCgjaQlejpAnIu0q-xtVfTEw_2026-09-01,0.00483054
UCOaUB8RzbLXB7QZtCmMO-8A_2026-09-01,0.02032231
UCgT-gb3favw2jQgwxbbwwaA_2026-09-01,0.00406663
UCos3GN2gX0wLcfuhOAiMotQ_2026-09-01,0.00440687
UCOLDn9gw0yegXRgjoi5MzNQ_2026-09-01,0.04111968
UC_8GOc5dWX2Kpb74_nQ7vSQ_2026-09-01,0.00265838
UCi8fe6P0_tLeZ1r31wOydvw_2026-09-01,0.18927084
UChLqHuZV0PENIBYxDgXyPLQ_2026-09-01,0.62090041
UChfs1dQ5phBDBWZJ1hHLbUA_2026-09-01,0.00505734
UCj5Y6sl23ln9PL6OiDTL7Gg_2026-09-01,0.43070814
UC_m_uqd_9s0hZvlgfvCzqbQ_2026-09-01,0.47351185
UC_9W6Z5Y3lDpLk47Y_y5ITQ_2026-09-01,0.02692725
UCoqMTw0yGmjIjNayukuOg5Q_2026-09-01,0.04745192
UCim7oePrB5hdfhcYwgUY6Jw_2026-09-01,0.01446681
UCcvKf8NxAw9i09hZw1MQPBQ_2026-09-01,0.00364635
UCjOtzpfj5lF9erwTEQYkxtQ_2026-09-01,0.00906752
UCxPMcxq7WJ32M3lnFn58Cpg_2026-09-01,0.00982203
UCPa8JSV9SPAjX7P2S-N5ssg_2026-09-01,0.00886269
UCPsZX-D7LW99PjZLA-UysPQ_2026-09-01,0.75958987
UCpnnUKmAQKgVinEV-lV-N2g_2026-09-01,0.0019713
UCtVBI7YLJyDvrx7iWAukx_Q_2026-09-01,0.00242476
UCvx8JsZvDVER74lG39dN9nw_2026-09-01,0.4677681
UCyEIUlSKbzCt3ElAsPFb7EQ_2026-09-01,0.17888511
UCVWAdUQGUiY2Kzh0JoB5PtQ_2026-09-01,0.03721831
UCUae69frfqpKOB3BUMdX5Qw_2026-09-01,0.00658264
UCdvY3MgO_BGrNbhbmxQC73Q_2026-09-01,0.73117188
UCWVS5IJOkNnkKS5rjiAt9rw_2026-09-01,0.49366028
UCp9skQ2mP_DfVL8c3gZQUzg_2026-09-01,0.74013799
UCz1Gc37nWsdF_jbJ_SvMl4g_2026-09-01,0.02723095
UCYM01qjjE_hmqFQAmOo-lNg_2026-09-01,0.74673387
UCXu_CcxBZA5S_iG33OzA9PQ_2026-09-01,0.76584821
UC-okJdotaXPlzivRzHFqzuA_2026-09-01,0.00379591
UCyZtdaxIiGb4wp5AmkEMrwQ_2026-09-01,0.30805633
UC0FUMakBJHsRrfKCGQuWnRA_2026-09-01,0.00364635
UCWwbZhw5M-Q5dZtTK4nDWNw_2026-09-01,0.0352072
UCYGK33LxFlgh9QtmvCYMR3A_2026-09-01,0.0169575
UCRE-9quk3OT2OKgZFAgdN_g_2026-09-01,0.06164324
UC0-ZCUCeCQBwKU15x0Aqk_g_2026-09-01,0.73343165
UCuaizlkqA5hZ_Dt3kAQkW3g_2026-09-01,0.02978325
UCVZ0SJNurJAv-6hV5VEGB2A_2026-09-01,0.0031225
UCX11f6rIVaoWGvFnKKyPBNQ_2026-09-01,0.48789862
UC-Pn1i_ot4qmG4l7wMZPvyw_2026-09-01,0.68601541
UCSrQnB-Ip9hc0biFQ3HoKzg_2026-09-01,0.03079895
UC141BuVRzSSXJX33Am9yxEA_2026-09-01,0.05470367
UC1q4Ihlv_YhLELw-ijE0Diw_2026-09-01,0.00239703
UC1aBdmKvC6yAiq4xgPLOzKA_2026-09-01,0.08860115
UCLJNL66b_LytPcZovqz5pkg_2026-09-01,0.11247497
UCnCFD3met1Ip7OngdF9TRDg_2026-09-01,0.01463209
UCSOeJWyByeoBHpmrm0NnlLw_2026-09-01,0.00416108
UC2_VPutZK0o_tG8ca_QrQRA_2026-09-01,0.04297668
UCIvLQ5lZ2kebqnLaGv_FP4w_2026-09-01,0.18315735
UC8aCmG_y4DqqVI9Xlcym1Jg_2026-09-01,0.60724227
UC5pdoCANExIVI4yf6DVoxew_2026-09-01,0.53395343
UCI3DTtB-a3fJPjKtQ5kYHfA_2026-09-01,0.41944015
UCjgTdV_oYP0TrrajTJLH33Q_2026-09-01,0.39716516
UCbvvdVOF0teLO-U6GWh1RKg_2026-09-01,0.19919161
UC6NXJtCn3p-actkvBUlHwDA_2026-09-01,0.01076068
UC8E29ANOC3_OuB7_5EPY2xw_2026-09-01,0.43920715
UCU0-6z_KU9AShyR9J6gcFGw_2026-09-01,0.26342506
UCQ5KMP03ifjJ8mDeyjWeZ7g_2026-09-01,0.15792149
UCRCK5FCJtomQT3b88jXI_DA_2026-09-01,0.25677012
UC59jWhPazXYi8Ny4vake45A_2026-09-01,0.00308682
UCT-UsvGz4gm7lezXvCEex9A_2026-09-01,0.55681409
UCN8YcyYvYIrtLcwyT-KqbNg_2026-09-01,0.01602297
UC9nWoDbRAl3-fvreBNeStKQ_2026-09-01,0.49654184
UCAhmOb9JWkI2HV_KIe9X_3A_2026-09-01,0.32047603
UCTUnpbuKoOODvhLhCO_kO_A_2026-09-01,0.39165857
UCZ-B_El841ME7qGXvYinGmw_2026-09-01,0.75747852
UC9vI-9WYQLXKI6n9XdXXo3g_2026-09-01,0.05776178
UCAGUSx1hbPKDqzBZH7bTjiA_2026-09-01,0.17552374
UCB4gIGaPUiLtpPreFSf8fKQ_2026-09-01,0.39992834
UCNRnCc0hg3ZcKdOGXKKv5Ww_2026-09-01,0.58224312
UCS2OAdHoLt-9T6cG9A2H49Q_2026-09-01,0.10684863
UCAVR6Q0YgYa8xwz8rdg9Mrg_2026-09-01,0.00567174
UCPpSXwAg3hkSpa85Ah-dpgA_2026-09-01,0.60448963
UCZHBZUrLKMoL6FxpvdRzhKA_2026-09-01,0.01088408
UC_e83kYlGHWbsrr8QCfV_ag_2026-09-01,0.04022021
UCBidluUUxzT_5jVOI_2x2MQ_2026-09-01,0.12078531
UCdT2nXooUR-X_MdmbER25yQ_2026-09-01,0.0132092
UCb08r5yPoH-AqM7_XCL4Uhg_2026-09-01,0.09048059
UCcNdwtZ7ulFFxngmnzsWQNA_2026-09-01,0.02847936
UCbItyvYa6cqvqtwlU2E9-wA_2026-09-01,0.06097982
UCcdlIcleb4oIK6of1ugSJ7w_2026-09-01,0.00607544
UCbIQGH_oFu0yfTWNPyquvpw_2026-09-01,0.3031639
UCkvz_CZlthtVo7E2sGa43cg_2026-09-01,0.00681269
UCCiKOocVSjyb2rsbL_Aanhw_2026-09-01,0.01513915
UCFHUplScMiWTjJrtv4hxWmg_2026-09-01,0.00231574
UCF7IBIvHZ2be4Ufo8i0uHBw_2026-09-01,0.02433826
UCFVRXOqBeh8ynDdYWztewXg_2026-09-01,0.75321815
UClWTMxooQefOLUV0TRcJ1Eg_2026-09-01,0.27018978
UCmZeuRfHYowZhf6odtoaNdg_2026-09-01,0.03367409
UClvMduw1MDMQ0PuH753-XSA_2026-09-01,0.11716072
UCGVy07wZs8W-EpKAGchrmCQ_2026-09-01,0.15639459
UCFsJVl9V9fSIZA1uS3gMu-w_2026-09-01,0.00392895
UC7SIaeUWuuIrWTIOBpayDVQ_2026-09-01,0.0983617
UCbFLnCfOet-u_kFyA7uX-9Q_2026-09-01,0.0115224
UCHYMpkGhNIr8A0m2H3lIjUA_2026-09-01,0.01028076
UC78d2aaqdrt02RfmtRLmlAw_2026-09-01,0.59896459
UC7ikVsVGGrUy2RD65w-awng_2026-09-01,0.11363083
UCCs1YZHxUTIbLZd-xrvvMCw_2026-09-01,0.0196451
UCpwGBltfwjd_fIzp5cb5R0w_2026-09-01,0.03560086
UCIOY-akTwtohcU0QB5ksjaA_2026-09-01,0.45344932
UC9DNu8JRIcE-rpo9eBx1ffw_2026-09-01,0.01233736
UCg5-nd3qoNnz5dzkX-O0b8A_2026-09-01,0.08143147
UCqRz_-l3GCmWb92u8P33RYQ_2026-09-01,0.66067023
UCroIk85IFzp6OZzKoJFDKuA_2026-09-01,0.19919161
UCqlcSsDNMYxEsem7bWY-66Q_2026-09-01,0.04541071
UCvfEv6nBnL0QlKqyGmL0orQ_2026-09-01,0.02224188
UCZFGZ_LhwgiTxaJzJ0QGdFw_2026-09-01,0.30073422
UCgWOZQvFZhGuqBCtjmee0XQ_2026-09-01,0.47063901
UC9bicYGdrx2YllCCg5H81Cw_2026-09-01,0.02055309
UCE8JC9RAVSudDT_8Ji1fuPQ_2026-09-01,0.00248117
UCD9tj7ATjYkyPyWXK_ewNXQ_2026-09-01,0.7916451
UCi_lv3g-CPdKF6nd68lvN3A_2026-09-01,0.44489335
UCUPa-gJcbYG9Ql8b9H9FBbg_2026-09-01,0.10252741
UC-pobd8lZE33BSDz-iyHfJQ_2026-09-01,0.22118232
UCYMXFMiaGBnZ08-wBHS2MPQ_2026-09-01,0.17719817
UC2k1bSUZ4X02jMs_sTS9jiQ_2026-09-01,0.05236767
UCN5oT4zGJX-_H6pE5isAEeg_2026-09-01,0.01639054
UCNrQ0VbdprXAur6OZKexjEw_2026-09-01,0.01254984
UC0uQYROl6H_8-Lb2bFz6yyg_2026-09-01,0.49077914
UC0Zl_lyMoTbroBSNv7SpGuQ_2026-09-01,0.00808825
UC2U8C7lmbQpfBeAW8fTP_jg_2026-09-01,0.05012613
UCL_IvtOZo6sONpiydbzzHpA_2026-09-01,0.00291444
UCKYdD83f363lIzcAwwhB-_A_2026-09-01,0.18315735
UCNpcMizRJ73OJm48LszuytQ_2026-09-01,0.23332595
UC2ja61Zku_rw3ZBfDZi8cuQ_2026-09-01,0.00262799
UCJA6MZLzTeljmGNFaugdoWQ_2026-09-01,0.54826424
UCKC4R6B8bTLIKH1V9PyD7Iw_2026-09-01,0.67851901
UCJN5mgLqkPCTVC5v-12gKjQ_2026-09-01,0.31299225
UCeAsj1O73pcqd-xQ4AZbJeQ_2026-09-01,0.52821208
UCSGKRGHGqP8W11G-SxsMaOw_2026-09-01,0.0512353
UCxVspLehsLr4bfqbJLFkS0w_2026-09-01,0.21920299
UCyz5WVrb-cz3myY04x_kVYw_2026-09-01,0.73567929
UCcoJiq6MyAqJeo3urxjDqrg_2026-09-01,0.66324969
UCyoKcrOcBEY4rmLYkDuXErA_2026-09-01,0.00288113
UCPp6kLLWggTuACsQwFibg8A_2026-09-01,0.01825282
UC_Cp4dXpd94km5Syq2x-xig_2026-09-01,0.76790902
UCZphffYmr6WuJD9SG8xwM8A_2026-09-01,0.06865775
UCcsC1X0SyiWcxBTbnjoZ9Ag_2026-09-01,0.42224976
UCFCt5RYu41HTg4eX1jdHNxw_2026-09-01,0.00472097
UCElXDMEQqlUnuyDbChjARWQ_2026-09-01,0.0831727
UC0-sEWFXIymAxx0InClynUA_2026-09-01,0.01899003
UCQePzo3_oRB-6RAzBioc6pg_2026-09-01,0.38072896
UC8chC5rneP-LHm43q-jF3oQ_2026-09-01,0.46489933
UCCTw9N950kEoELFv3dRcWJQ_2026-09-01,0.05067785
UCj7wKA4vwdzMP35lO5CCwgQ_2026-09-01,0.33572123
UC8IxyKkhu4YUYK_c76AdcDw_2026-09-01,0.10575352
UCq6bX1rAJqwt3LmYuHrDPqg_2026-09-01,0.69095994
UCABvetlXpdOwFz10fCpJkwQ_2026-09-01,0.00420913
UC_Xb6sEXAdNv9nHU4988vmg_2026-09-01,0.29112767
UCFJAOgvqcc3fPqdu7fTl-zA_2026-09-01,0.09938877
UC3NIb2qVtwunRNmypsDpQzA_2026-09-01,0.03256675
UCg5Ew4lj74r_ohbc4AeQj1w_2026-09-01,0.04642064
UC6LLMkdyses2ugPRNEXutMQ_2026-09-01,0.12704113
UCe0loHR-Vltq0GPHgYyzmrQ_2026-09-01,0.82547493
UCADf92tisOOh90hrZee4-LA_2026-09-01,0.13762489
UCeiQKedLZob9Cb797XfsrRw_2026-09-01,0.14178092
UCCoU5ViXgoK-_2neelNhQaA_2026-09-01,0.00250986
UCIxcZWLQpP-mUZC6_Uljc_Q_2026-09-01,0.0329319
UCRLoyYlPR9ufzlbeI9lLDqg_2026-09-01,0.10359297
UC1dHu9GhbHH7RcHKyJdaOvA_2026-09-01,0.00523436
UCWq9wRjQXYC8i486uVLysUA_2026-09-01,0.00268912
UCyaMGpNb-UiLHMY833ai7IQ_2026-09-01,0.02249395
UCybn_0S0Ec6Il1ZhiQM1Vww_2026-09-01,0.02326703
UCk1NTIlGP27bQEumkw6sDVA_2026-09-01,0.08675703
UCkNUuK9EfixaK2HbJYZ5YrQ_2026-09-01,0.0380533
UCoIEwIHhW-1_roLnb5nK_UA_2026-09-01,0.53682083
UCHJv9xBR9lkzGSIepie-T_A_2026-09-01,0.01351311
UCfHlKp8w7IxpObWnHJtYRnQ_2026-09-01,0.32805331
UCG3uX-OwJ82T3L3EhC3GPXA_2026-09-01,0.60998811
UCqFmG6ofirSWvI9B8mwzvyA_2026-09-01,0.8304011
UCXg-6b0X0sIMBDHhbyBuOQg_2026-09-01,0.03977757
UC9qwDE50UcXvdplRNZKEoPA_2026-09-01,0.39440851
UCTbwgCCBrI-4XU8b5B0uGPg_2026-09-01,0.00488628
UCVqmxk6oNOHj9RFVo4uP_lw_2026-09-01,0.67599937
UCXNh3eUHXEkL7h1w1KYQRKQ_2026-09-01,0.82212856
UCIH7bdyhaJ0nD7jIcgqmQbQ_2026-09-01,0.21723643
UCTizVc_OD9jdhWZOnrQnUeQ_2026-09-01,0.37260967
UC-ntxe7eU_afJWEpcTNlVFA_2026-09-01,0.02662685
UC0vqyzJ-1l3AJl3IkKu0bhQ_2026-09-01,0.36456161
UCJNEIbzzpRl-l1iGdD2BxGg_2026-09-01,0.54540769
UCLbehOwlIi6QrfZNhC39t4Q_2026-09-01,0.06231341
UC-DpK48x4bltMjHHChZ4zBg_2026-09-01,0.19463448
UCTDBdmaY4m3ipZaLiYA6FWg_2026-09-01,0.02545752
UCaCFlaIQAmiU_mLnZq5E-tA_2026-09-01,0.36723614
UCRltQqhqu9nXMRLaX0f8Cow_2026-09-01,0.07972352
UCOwgg4IVhPQtq6qtxEeX3XA_2026-09-01,0.40269791
UChfA8Yu8dk5k78UzaM6tsRw_2026-09-01,0.03443242
UCjiXXUsdl0F0aUkTbv8ZVTA_2026-09-01,0.78197848
UCah0w9n_eAyjUBG7eHbj8vw_2026-09-01,0.01430338
UCEGtfhdJiRgH98mVMEQZThQ_2026-09-01,0.12325512
UCqEoc6yuYRg5CFG4NIG8euQ_2026-09-01,0.0260358
UCEebX3m88Arwc91LPmEmTYQ_2026-09-01,0.01063866
UCHZ9X2Nl3dq8ChEJgScmZCw_2026-09-01,0.01715073
UCtjlhCmFXYFXC4uqg1flTBg_2026-09-01,0.52246325
UCWZqSCSNZT3X8e_HyCJoXMg_2026-09-01,0.16733816
UC6aF-skLT_1X6cdPJxVJXxQ_2026-09-01,0.00636031
UCApYlKNymyHZ9dlOCImFVBg_2026-09-01,0.16257489
UClHGjnruf4A9Us7CHVgLIEQ_2026-09-01,0.5681609
UCWKOw78U77xw8alsPSW_AeA_2026-09-01,0.25022618
UCNZiNM2pP6uhIlTQVS2Q5fA_2026-09-01,0.00541755
UCm1JuixwDoQEDi5RfcRuHJg_2026-09-01,0.01205957
UCAgZP6qUjcVjZsGWtW-l6lw_2026-09-01,0.46203288
UCLuDznROytXtnMSDwo5aGJQ_2026-09-01,0.01414176
UC3kVT7zilzGf_7fIEkc4dDA_2026-09-01,0.00621626
UCsbo5VemDR505Etd5ZKnWNA_2026-09-01,0.1610116
UCuetAJhNx1unKTXo7e1c0xQ_2026-09-01,0.34088205
UCdtV_sB8WUWo-P_q3OwNfcw_2026-09-01,0.6580812
UCNOj6xtKWCtPbjJV8XPlbLg_2026-09-01,0.03481773
UCs7mzshFv0PbUMUM0x1Waug_2026-09-01,0.04903973
UCkrJBQAut47U8xMUuv8D_3g_2026-09-01,0.01986835
UC4OsulKlyGi0wY5h9D4JjBw_2026-09-01,0.57380835
UCb3wpKfVlnPx2bym2Tivbzw_2026-09-01,0.26566783
UCxtZZzggcfOmsW99gYkxsNA_2026-09-01,0.05713759
UCrdRt77s53QWevhMYX-8UPg_2026-09-01,0.01754371
UCop7QCLcdzTpcYZzMu1mFAg_2026-09-01,0.0772229
UCppLE7C1TC4ovrGbRoTzsmg_2026-09-01,0.35923768
UCulcJNJPF25ncT7GvJm6iEg_2026-09-01,0.26792276
UCjk9szhhsAO76PdH450mZNg_2026-09-01,0.29351232
UCd4PDK0JYfPf0DUeYZThguw_2026-09-01,0.14038406
UCm-Yd-aYSagMqVdMZSYiKLQ_2026-09-01,0.14318936
UCAZVLirygDNDfCNfyIVnTWA_2026-09-01,0.1705752
UCZX4g_FvyOfyTIKxbMfKfIg_2026-09-01,0.10466833
UCOKQpmfKrtu7Nu719CVSw1A_2026-09-01,0.1533769
UC-2JsFNClcIesgd7UmtGnpw_2026-09-01,0.27246884
UCijUV3SuVe_qErzBpdOmKIA_2026-09-01,0.14460943
UC07QfC3yXeYr2SfIL6WXpOw_2026-09-01,0.41383674
UCETJ2rkwgeYb0OojPNh-DRw_2026-09-01,0.11019402
UCCrqkpYfyGX5UBFyPx6bqKA_2026-09-01,0.65548271
UCD7D_WRXbei_g6xXOZIhQnQ_2026-09-01,0.05352364
UClSuqiIAACDUJrH3gb4JcLQ_2026-09-01,0.01496824
UCs4CTQE-t9kj-6Xn3Cjtf-A_2026-09-01,0.51670847
UCL3gnarNIeI_M0cFxjNYdAA_2026-09-01,0.06719812
UCzju3TlIRi6tkO9o4gwXdVA_2026-09-01,0.3539482
UCerhqgFd8dnypewC3yjAlbg_2026-09-01,0.07888187
'''
try_shot2 = clean(pd.read_csv(io.StringIO(TRY_SHOT2_CSV)))
print('내장 try_shot2:', try_shot2.shape)

## 1. try_shot2 블렌드 (blend_t01.py 와 동일 공식)
`rank = rankpct(Σ w_i · rankpct(src_i))`, `pred = 1/(1+exp(-8(rank-0.8)))`

In [ ]:
# 1단계: 세 모델 출력이 있으면 직접 블렌딩, 없으면 내장된 try_shot2 사용
if all(os.path.exists(p) for p in [MATE, T01, QJ]):
    def rankpct(path):
        s = clean(pd.read_csv(path)).set_index('row_id').prediction.reindex(order)
        s = s.fillna(s.min())
        return s.rank(pct=True).values
    W = {'mate': 0.70, 't01': 0.20, 'qj': 0.10}
    score = W['mate']*rankpct(MATE) + W['t01']*rankpct(T01) + W['qj']*rankpct(QJ)
    rr = pd.Series(score).rank(pct=True).values
    pred = 1/(1+np.exp(-8*(rr-0.8)))
    sub = pd.DataFrame({'row_id': order, 'prediction': np.round(pred, 8)})
    sub.head()
    print('세 모델 출력으로 블렌딩')
else:
    sub = try_shot2.set_index('row_id').reindex(order).reset_index()
    print('내장 try_shot2 사용 (세 모델 출력 파일 없음)')
sub.head()

## 2. 규칙: 지난주 정답 1 + 관측 구간(08-25~08-31) 롱폼 0편 → 최상위 (원래 순서 유지)

In [ ]:
v = clean(pd.read_csv(os.path.join(DATA, 'video_5d_views.csv')))
v['published_at'] = pd.to_datetime(v.published_at, utc=True).dt.tz_localize(None)
L = v[(v.is_shorts == 0) & v.view_5d.notna()]
n_obs = L[(L.published_at >= D_PREV) & (L.published_at < D_TEST)].groupby('channel_id').size()
tl = clean(pd.read_csv(os.path.join(DATA, 'train_labels.csv')))
tl = tl[tl.row_id.astype(str).str.endswith(D_PREV)]
prev = pd.Series(tl.target.values, index=tl.row_id.str[:-len(D_PREV)-1])
ch = sub.row_id.str[:-len(D_TEST)-1]
A = ch.map(prev).eq(1) & ch.map(n_obs).fillna(0).eq(0)
print('규칙 대상 채널 수:', int(A.sum()))   # 9
r = sub.prediction.rank(ascending=False, method='first')
mx = sub.prediction.max(); ra = r[A]
sub.loc[A, 'prediction'] = np.minimum(0.99999, mx + (1-mx)*0.5*(1-(ra-ra.min())/(ra.max()-ra.min()+1)*0.5))
print('대상 채널 새 순위:', sorted(sub.prediction.rank(ascending=False, method='first')[A].astype(int)))

## 3. 저장 및 형식 점검

In [ ]:
OUT = '5th_datathon_submission_빡빡이.csv'
sub.to_csv(OUT, index=False, encoding='utf-8', lineterminator='\n')
chk = pd.read_csv(OUT)
assert len(chk) == len(order) and list(chk.columns) == ['row_id', 'prediction']
assert chk.prediction.between(0, 1).all() and chk.prediction.notna().all() and not chk.row_id.duplicated().any()
print('OK', OUT, len(chk))

## 부록: 구성 모델 재현
- **q_joint**: `python run_final.py --data ./epoch_data --F 26 --prev_F 18 --lag1feat --lag1q --q_joint --mode lb7627 --q_x --soft_k --out a.csv` 와 `--mode latest --out b.csv` 의 순위 반반 (`pct=True`)
- **sim (mate의 80%)**: `python run_sim.py --S 8000 --seed 0` → `sim_p.csv`
- **mate** = 0.8·rank(sim) + 0.2·rank(T01)
- 검증: 시간 기반(데이터가 검증일에 끝난다고 놓고 소프트 라벨까지 재생성). KFold 사용 안 함.